# Extracción de metadatos de estudios
Este notebook lee imágenes de diferentes estudios y extrae sus respectivos metadatos para exportarlos en un archivo de Excel

Elaborado por Juan Manuel Rivera/AJS

# Importación de librerías y funciones auxiliares

In [ ]:
import os
import pandas as pd
import pydicom
import re
import numpy as np

En la siguiente celda está la dirección del directorio donde están las imágenes

In [ ]:
path = 'C:/Users/ant-sala/OneDrive - Universidad de los andes/TAC/IMAGENES/FSFB166'
#path = '.'

In [ ]:
def list_one_image_per_folder(path):
    images_path = []
    for item in os.listdir(path):
        item_path = f'{path}/{item}'
        if os.path.isdir(item_path):
            images_path += list_one_image_per_folder(item_path)
        elif os.path.isfile(item_path) and '.dcm' in item:
            images_path.append(item_path)
            break
        else:
            print(f'Error al procesar el documento {item_path}')
    return images_path

In [ ]:
def list_dicom_image_folders(path):
    images_path = []
    for item in os.listdir(path):
        item_path = f'{path}/{item}'
        if os.path.isdir(item_path):
            images_path += list_one_image_per_folder(item_path)
        elif os.path.isfile(item_path) and '.dcm' in item:
            images_path.append(path)
            break
        else:
            print(f'Error al procesar el documento {item_path}')
    return images_path

# Extracción de metadatos

Si desea añadir algún metadato añada el DICOM Tag en la siguiente lista

In [ ]:
tags = {
    # File path
    "file_path" : [],
    # Mean z-pacing
    "mean_spacing" : [],
    # standard deviarion of z-pacing
    "sd_spacing" : [],
    # Number of slices
    "n_slices" : [],
    # Pixel Spacing
    "00280030" : [],
    # Slice thickness
    "00180050" : [],
    # Institution name
    "00080080" : [],
    # Manufacturer
    "00080070" : [],
    # Manufacturer model
    "00081090" : [],
    # kiloVolt_peak
    "00180060" : [],
    # Exposure time
    "00181150" : [],
    # Xray tube current
    "00181151" : [],
    # Exposure mAs
    "00181152" : [],
    # Protocol name
    "00181030" : [],
    # Series description
    "0008103E" : [],
    # Patient ID
    "00100020" : [],
    # Accsesion number
    "00080050" : [],
    # Patient age
    "00101010" : [],
    # Patient sex
    "00100040" : []
}

In [ ]:
# dicom_images_paths = list_one_image_per_folder(path)
dicom_folders = list_dicom_image_folders(path)
print(f'Se van a procesar {len(dicom_folders)} estudios')

In [ ]:
for img_path in dicom_folders:
    tags_to_read = True
    z_positions = []
    for file in os.listdir(img_path):
        img = pydicom.dcmread(os.path.join(img_path, file))
        if tags_to_read:
            for tag, array in tags.items():
                if re.fullmatch("\d{8}", tag):
                    array.append(img[tag].value)
                elif tag == "file_path":
                    array.append(img_path)
                elif tag in ["mean_spacing", "sd_spacing", "n_slices"]:
                    pass
                else:
                    array.append('')
            tags_to_read = False
        z_positions.append(img['00200032'].value)
    z_positions = sorted(z_positions)
    diffs = np.diff(z_positions)
    tags["mean_spacing"].append(np.mean(diffs))
    tags["sd_spacing"].append(np.std(diffs))
    tags["n_slices"].append(len(z_positions))

In [ ]:
df = pd.DataFrame(data=tags)
df.rename(columns={
    "file_path" : "Dirección del estudio",
    "mean_spacing" : "Media de diatancia z-axis",
    "sd_spacing" : "Desviación estándar de diatancia z-axis",
    "n_slices" : "Número de cortes",
    "00280030" : "Pixel spacing (mm)",
    "00180050" : "Slice thickness (mm)",
    "00080080" : "Institution name",
    "00080070" : "Manufacturer",
    "00081090" : "Manufacturer model",
    "00180060" : "Peak kilo voltage output",
    "00181150" : "Exposure time (miliseconds)",
    "00181151" : "Xray tube current (mA)",
    "00181152" : "Exposure (mAs)",
    "00181030" : "Protocol name",
    "0008103E" : "Series description",
    "00100020" : "Patient ID",
    "00080050" : "Accsesion number",
    "00101010" : "Patient age",
    "00100040" : "Patient sex"
    },
    inplace=True
    )
display(df.head(5))

In [ ]:
df.to_excel("MetadatosAISD.xlsx")